# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [21]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)

con = duckdb.connect()

try:
    con.execute("DROP SECRET IF EXISTS hf_secret")
except:
    pass

con.execute(f"""
CREATE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

HF_DATASET = "hf://datasets/FlyRank/internship-warehouse"

print("DuckDB connected")
print("Hugging Face configured")

Token loaded: True
DuckDB connected
Hugging Face configured


In [22]:
con.sql(f"""
SELECT *
FROM read_parquet(
    '{HF_DATASET}/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
LIMIT 5
""").show()

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬───────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_clau

In [23]:
content_columns = con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet(
    '{HF_DATASET}/dim_content.parquet'
)
""").df()

print(content_columns["column_name"].tolist())

['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']


In [24]:
march_content = con.sql(f"""
SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.report_date,
    f.gsc_impressions,
    f.gsc_clicks,
    f.gsc_avg_position,
    c.content_updated_date,
    DATE_DIFF(
        'day',
        CAST(c.content_updated_date AS DATE),
        f.report_date
    ) AS days_since_update
FROM read_parquet(
    '{HF_DATASET}/fact_content_daily_performance/month=2026-03/data_0.parquet'
) f
LEFT JOIN read_parquet(
    '{HF_DATASET}/dim_content.parquet'
) c
ON f.client_hash_id = c.client_hash_id
AND f.content_hash_id = c.content_hash_id
""")

print("Rows:", len(march_content))
print("Rows:", len(march_content))
march_content.limit(5).show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 9841378


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 9841378
┌─────────────────────────┬──────────────────────────┬─────────────┬─────────────────┬────────────┬───────────────────┬──────────────────────┬───────────────────┐
│     client_hash_id      │     content_hash_id      │ report_date │ gsc_impressions │ gsc_clicks │ gsc_avg_position  │ content_updated_date │ days_since_update │
│         varchar         │         varchar          │    date     │      int64      │   int64    │      double       │         date         │       int64       │
├─────────────────────────┼──────────────────────────┼─────────────┼─────────────────┼────────────┼───────────────────┼──────────────────────┼───────────────────┤
│ client_73cda7b4e4f265ea │ content_b7e512995f79d5a6 │ 2026-03-01  │              20 │          0 │              3.35 │ 2026-05-18           │               -78 │
│ client_73cda7b4e4f265ea │ content_05597932fe4da067 │ 2026-03-01  │               1 │          0 │               0.0 │ 2026-05-18           │               -78 │
│ client

In [25]:
staleness_check = con.sql("""
SELECT
    CASE
        WHEN days_since_update < 90 THEN '0-89 days'
        WHEN days_since_update < 180 THEN '90-179 days'
        WHEN days_since_update < 365 THEN '180-364 days'
        ELSE '365+ days'
    END AS staleness_bucket,
    COUNT(*) AS n,
    ROUND(AVG(gsc_impressions), 2) AS avg_impressions,
    ROUND(AVG(gsc_clicks), 2) AS avg_clicks
FROM march_content
WHERE days_since_update >= 0
GROUP BY 1
ORDER BY
    CASE
        WHEN staleness_bucket = '0-89 days' THEN 1
        WHEN staleness_bucket = '90-179 days' THEN 2
        WHEN staleness_bucket = '180-364 days' THEN 3
        ELSE 4
    END
""")

staleness_check.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────┬────────┬─────────────────┬────────────┐
│ staleness_bucket │   n    │ avg_impressions │ avg_clicks │
│     varchar      │ int64  │     double      │   double   │
├──────────────────┼────────┼─────────────────┼────────────┤
│ 0-89 days        │ 944905 │           35.98 │       0.08 │
│ 90-179 days      │ 134754 │            3.52 │       0.01 │
│ 180-364 days     │  89614 │            0.19 │        0.0 │
└──────────────────┴────────┴─────────────────┴────────────┘



### Signal 1 — Staleness

**Signal:** Days since the content was last updated.

**Why it matters:** Staleness is linked to FlyRank's refresh-flag logic, so it is a relevant signal to audit for a content-refresh baseline.

**Verdict: CONFIRMED**

**Observation:** Older content shows substantially lower average impressions and clicks in this March 2026 slice. Average impressions decrease from 35.98 for content updated within 0–89 days to 0.19 for content that is 180–364 days old. This is a directional observation, not a claim that staleness causes lower traffic.

In [26]:
ctr_position_check = con.sql("""
SELECT
    CASE
        WHEN gsc_avg_position <= 3 THEN '1-3'
        WHEN gsc_avg_position <= 10 THEN '4-10'
        WHEN gsc_avg_position <= 20 THEN '11-20'
        ELSE '21+'
    END AS position_bucket,

    COUNT(*) AS n,

    ROUND(
        AVG(
            CASE
                WHEN gsc_impressions > 0
                THEN CAST(gsc_clicks AS DOUBLE) / gsc_impressions
                ELSE NULL
            END
        ),
        4
    ) AS avg_ctr

FROM march_content

WHERE gsc_avg_position IS NOT NULL
  AND gsc_impressions > 0

GROUP BY 1

ORDER BY
    CASE
        WHEN position_bucket = '1-3' THEN 1
        WHEN position_bucket = '4-10' THEN 2
        WHEN position_bucket = '11-20' THEN 3
        ELSE 4
    END
""")

ctr_position_check.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬─────────┬─────────┐
│ position_bucket │    n    │ avg_ctr │
│     varchar     │  int64  │ double  │
├─────────────────┼─────────┼─────────┤
│ 1-3             │  727362 │  0.0048 │
│ 4-10            │ 1456122 │  0.0035 │
│ 11-20           │  519223 │  0.0028 │
│ 21+             │  908354 │  0.0013 │
└─────────────────┴─────────┴─────────┘



### Signal 2 — CTR vs Position

**Signal:** Click-through rate (CTR) relative to average search position.

**Why it matters:** CTR relative to position is linked to FlyRank's CTR-fix logic, so it is a useful signal to audit.

**Verdict: CONFIRMED**

**Observation:** Average CTR decreases as the position bucket gets worse. CTR is 0.0048 for positions 1–3 and decreases to 0.0013 for positions 21+. This is a directional observation and does not by itself establish causation.

In [27]:
baseline = con.sql("""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    content_updated_date,
    days_since_update,

    CASE
        WHEN days_since_update >= 180 THEN 2
        WHEN days_since_update >= 90 THEN 1
        ELSE 0
    END AS stale_score,

    CASE
        WHEN gsc_avg_position > 20
             AND gsc_impressions > 0
             AND (CAST(gsc_clicks AS DOUBLE) / gsc_impressions) < 0.002
        THEN 2

        WHEN gsc_avg_position > 10
             AND gsc_impressions > 0
             AND (CAST(gsc_clicks AS DOUBLE) / gsc_impressions) < 0.003
        THEN 1

        ELSE 0
    END AS ctr_position_score

FROM march_content
WHERE gsc_impressions > 0
""")

baseline.limit(5).show()

┌─────────────────────────┬──────────────────────────┬─────────────┬─────────────────┬────────────┬───────────────────┬──────────────────────┬───────────────────┬─────────────┬────────────────────┐
│     client_hash_id      │     content_hash_id      │ report_date │ gsc_impressions │ gsc_clicks │ gsc_avg_position  │ content_updated_date │ days_since_update │ stale_score │ ctr_position_score │
│         varchar         │         varchar          │    date     │      int64      │   int64    │      double       │         date         │       int64       │    int32    │       int32        │
├─────────────────────────┼──────────────────────────┼─────────────┼─────────────────┼────────────┼───────────────────┼──────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ client_73cda7b4e4f265ea │ content_b7e512995f79d5a6 │ 2026-03-01  │              20 │          0 │              3.35 │ 2026-05-18           │               -78 │           0 │                  0 │
│ client_7

In [28]:
baseline_scored = con.sql("""
SELECT
    *,
    stale_score + ctr_position_score AS score
FROM baseline
ORDER BY score DESC
""")

baseline_scored.limit(10).show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬──────────────────────────┬─────────────┬─────────────────┬────────────┬────────────────────┬──────────────────────┬───────────────────┬─────────────┬────────────────────┬───────┐
│     client_hash_id      │     content_hash_id      │ report_date │ gsc_impressions │ gsc_clicks │  gsc_avg_position  │ content_updated_date │ days_since_update │ stale_score │ ctr_position_score │ score │
│         varchar         │         varchar          │    date     │      int64      │   int64    │       double       │         date         │       int64       │    int32    │       int32        │ int32 │
├─────────────────────────┼──────────────────────────┼─────────────┼─────────────────┼────────────┼────────────────────┼──────────────────────┼───────────────────┼─────────────┼────────────────────┼───────┤
│ client_65de48885f4ef01b │ content_95d140b7cd7e3897 │ 2026-03-01  │               4 │          0 │              35.25 │ 2025-07-31           │               213 │         

In [29]:
final_queue = con.sql("""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    days_since_update,
    score,

    CASE
        WHEN stale_score >= 2
             AND ctr_position_score >= 2
            THEN 'STALE_AND_WEAK_CTR'

        WHEN stale_score >= 2
            THEN 'STALE_CONTENT'

        WHEN ctr_position_score >= 2
            THEN 'WEAK_CTR_FOR_POSITION'

        ELSE 'LOW_PRIORITY'
    END AS reason_code,

    CASE
        WHEN score >= 3 THEN 'REFRESH'
        WHEN score = 2 THEN 'REVIEW'
        ELSE 'MONITOR'
    END AS action

FROM baseline_scored

ORDER BY score DESC
""")

final_queue.limit(10).show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬──────────────────────────┬─────────────┬─────────────────┬────────────┬────────────────────┬───────────────────┬───────┬────────────────────┬─────────┐
│     client_hash_id      │     content_hash_id      │ report_date │ gsc_impressions │ gsc_clicks │  gsc_avg_position  │ days_since_update │ score │    reason_code     │ action  │
│         varchar         │         varchar          │    date     │      int64      │   int64    │       double       │       int64       │ int32 │      varchar       │ varchar │
├─────────────────────────┼──────────────────────────┼─────────────┼─────────────────┼────────────┼────────────────────┼───────────────────┼───────┼────────────────────┼─────────┤
│ client_65de48885f4ef01b │ content_95d140b7cd7e3897 │ 2026-03-01  │               4 │          0 │              35.25 │               213 │     4 │ STALE_AND_WEAK_CTR │ REFRESH │
│ client_65de48885f4ef01b │ content_95d140b7cd7e3897 │ 2026-03-03  │               5 │          0 │ 

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [30]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [31]:
import os

os.makedirs("work/outputs", exist_ok=True)

print("Folder ready:", os.path.exists("work/outputs"))

Folder ready: True


In [32]:
con.execute("""
COPY (
    SELECT *
    FROM final_queue
)
TO 'work/outputs/baseline_action_score.csv'
WITH (HEADER, DELIMITER ',')
""")

print("Queue written successfully.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Queue written successfully.


In [33]:
import os

print(
    "File exists:",
    os.path.exists("work/outputs/baseline_action_score.csv")
)

File exists: True


In [34]:
top10 = con.sql("""
SELECT *
FROM final_queue
ORDER BY score DESC
LIMIT 10
""")

top10.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬──────────────────────────┬─────────────┬─────────────────┬────────────┬────────────────────┬───────────────────┬───────┬────────────────────┬─────────┐
│     client_hash_id      │     content_hash_id      │ report_date │ gsc_impressions │ gsc_clicks │  gsc_avg_position  │ days_since_update │ score │    reason_code     │ action  │
│         varchar         │         varchar          │    date     │      int64      │   int64    │       double       │       int64       │ int32 │      varchar       │ varchar │
├─────────────────────────┼──────────────────────────┼─────────────┼─────────────────┼────────────┼────────────────────┼───────────────────┼───────┼────────────────────┼─────────┤
│ client_65de48885f4ef01b │ content_95d140b7cd7e3897 │ 2026-03-01  │               4 │          0 │              35.25 │               213 │     4 │ STALE_AND_WEAK_CTR │ REFRESH │
│ client_65de48885f4ef01b │ content_95d140b7cd7e3897 │ 2026-03-02  │               5 │          0 │ 

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 3) Top-10 Review

The baseline ranks content for refresh using staleness and CTR relative to search position.

For each top-ranked item, I review:
- the recommended action,
- why the rule selected it,
- what could make the recommendation wrong.

The top ten items all received the maximum score of 4 because they are both stale (180+ days) and have weak CTR relative to their search position.

In [35]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top10_df = top10.df()

top10_df[
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "days_since_update",
        "score",
        "reason_code",
        "action"
    ]
]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,days_since_update,score,reason_code,action
0,client_73cda7b4e4f265ea,content_3e435be3dc7de7ef,2,0,92.000000,216,4,STALE_AND_WEAK_CTR,REFRESH
1,client_73cda7b4e4f265ea,content_bae9e4d389512ca6,6,0,47.000000,205,4,STALE_AND_WEAK_CTR,REFRESH
2,client_73cda7b4e4f265ea,content_836b85ddd3b20839,1,0,97.000000,208,4,STALE_AND_WEAK_CTR,REFRESH
3,client_b10cb2997d0c7c86,content_f406ec8774d96559,5,0,41.400000,206,4,STALE_AND_WEAK_CTR,REFRESH
4,client_73cda7b4e4f265ea,content_137163d1927eb1c2,1,0,24.000000,209,4,STALE_AND_WEAK_CTR,REFRESH
5,client_73cda7b4e4f265ea,content_f4098d5b2c2eeb18,4,0,77.250000,216,4,STALE_AND_WEAK_CTR,REFRESH
6,client_73cda7b4e4f265ea,content_836b85ddd3b20839,4,0,57.750000,209,4,STALE_AND_WEAK_CTR,REFRESH
7,client_73cda7b4e4f265ea,content_6d38a1d4a185b93c,1,0,84.000000,217,4,STALE_AND_WEAK_CTR,REFRESH
8,client_73cda7b4e4f265ea,content_f4098d5b2c2eeb18,4,0,111.000000,217,4,STALE_AND_WEAK_CTR,REFRESH
9,client_65de48885f4ef01b,content_95d140b7cd7e3897,9,0,23.888889,218,4,STALE_AND_WEAK_CTR,REFRESH


| Rank | Action | Why it's here | What would make it wrong |
|---|---|---|---|
| 1 | REFRESH | Stale for 213 days and has 4 impressions with 0 clicks and position 35.25. | The page may have very little search demand, so refreshing it may not produce meaningful gains. |
| 2 | REFRESH | Stale for 215 days with 5 impressions, 0 clicks, and position 54.4. | Extremely low impressions suggest the problem may be demand or discoverability rather than freshness. |
| 3 | REFRESH | Stale for 216 days with 2 impressions, 0 clicks, and position 92.0. | The page ranks very poorly and has almost no impressions, so a refresh alone may not help. |
| 4 | REFRESH | Stale for 216 days with 4 impressions, 0 clicks, and position 77.25. | Low visibility may mean that refreshing content is not the main lever. |
| 5 | REFRESH | Stale for 208 days with 1 impression, 0 clicks, and position 97.0. | Almost no search visibility makes this a weak candidate despite the high rule score. |
| 6 | REFRESH | Stale for 205 days with 6 impressions, 0 clicks, and position 47.0. | The very low impression volume may indicate limited search demand. |
| 7 | REFRESH | Stale for 206 days with 5 impressions, 0 clicks, and position 41.4. | The rule does not know whether the topic is still strategically important. |
| 8 | REFRESH | Stale for 278 days with 2 impressions, 0 clicks, and position 71.5. | Very low impressions and poor ranking could mean the page needs a different intervention than a refresh. |
| 9 | REFRESH | Stale for 278 days with 4 impressions, 0 clicks, and position 42.5. | The rule cannot distinguish low-quality content from pages with naturally low demand. |
| 10 | REFRESH | Stale for 219 days with 1 impression, 0 clicks, and position 97.0. | Almost no visibility means there is little evidence that refreshing the page will improve traffic. |

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [36]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4) Weak Picks

The baseline has a clear weakness: several top-ranked pages have extremely low impressions.

Although these pages satisfy the staleness and weak-CTR conditions, their very low search visibility means that a refresh may not be the best action. The rule does not account for search demand, content importance, or whether the page has enough traffic opportunity to justify the work.

This suggests that a future model could improve on the baseline by considering additional observable signals such as search volume and stronger measures of traffic opportunity.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.